# KíLín performance study from diagnostic traces

This notebook is a **baseline study**, not a speed optimization. It
analyzes KíLín's existing, numbered conversation-trace JSON files and can
optionally ask a **fixed set of questions** in an isolated local KíLín
workspace. It uses no new timing hooks in KíLín's production code.

The default run is **read-only**: it does not load models, create files, or
modify projects. Change the options below explicitly to run a benchmark,
export timing-only CSV files, or perform a short writer-cost check.

**Privacy:** Input traces can contain full prompts, retrieved text, model
outputs, and token IDs. Keep them private. The analysis reads these files but
returns only operation names, numeric durations, counts, and run labels.
Never upload raw conversation traces as benchmark results.

## 1. Set up the repository and imports

Run from a notebook kernel in the KíLín environment. No optional plotting or
data-analysis packages are required. Model dependencies are needed only if
you enable the live benchmark.

In [ ]:
from __future__ import annotations

import csv
from configparser import ConfigParser
import os
from pathlib import Path
import platform
import sys
from tempfile import TemporaryDirectory
from time import perf_counter


def find_repository_root(start: Path | None = None) -> Path:
    """Locate the checkout containing the KíLín package and assets.

    Parameters
    ----------
    start: pathlib.Path | None, optional
        Directory used as the starting point, by default the kernel's cwd.

    Returns
    -------
    pathlib.Path
        Canonical path to the repository root.

    Raises
    ------
    RuntimeError
        If the notebook is not running beneath the KíLín checkout.

    """

    current = (start or Path.cwd()).resolve()
    for directory in (current, *current.parents):
        if (directory / "kilin").is_dir() and (
            directory / "assets" / "trace_benchmark.py"
        ).is_file():
            return directory
    raise RuntimeError("Launch the notebook inside a KíLín checkout.")


repository_root = find_repository_root()
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from assets.trace_benchmark import (
    question_statistics,
    read_trace_operations,
    summarize_trace_turns,
)

print("Platform:", platform.system(), platform.machine())
print("Python:", platform.python_version())

## 2. Configure the study

Start by analyzing saved traces with `EXISTING_TRACES`. Use arbitrary,
**non-sensitive labels** as the dictionary keys. A single conversation trace
may contain multiple turns and multiple logger sessions after reopening.

Live benchmarking is **disabled by default**. To enable it, create a *separate
benchmark KíLín workspace* inside `sandbox/`, with its own `profile.ini` and
source corpus. Set `[models] allow_download = false` and make sure both models
are already cached. The notebook rejects an external or download-enabled
workspace. This prevents accidental mutation of your working projects.

`USE_ROUTER = False` selects the project type's default intent. This gives
repeatable RAG-path measurements without routing or tool selection. Enable
routing explicitly if that is the behavior you want to study; the questions
should remain non-tool requests. The question order and conversation history
are the same for every repetition, but later questions still see earlier turns.

In [ ]:
RUN_BENCHMARK = False
BENCHMARK_PROJECT = repository_root / "sandbox" / "performance_workspace"
QUESTION_SET = "code"  # "code" or "literature"
REPEATS = 2
TRACE_SETTINGS = (True,)  # Use (True, False) for a small overhead comparison.
USE_ROUTER = False

# Explicit existing file paths, not a recursive scan of project logs.
EXISTING_TRACES = {
    # "previous_run": Path("/absolute/path/to/conversation.json"),
}

EXPORT_CSV = False
TRACE_WRITE_CHECK = False
RESULTS_DIRECTORY = repository_root / "sandbox" / "performance_benchmark"

## 3. Fixed question sets

The two sets test comparable *roles* (overview, mechanism, limitations), not
identical factual content. Keep the text and order fixed when comparing runs.
These are generic questions, so answer correctness is **not** being evaluated.
Question IDs, rather than full questions, appear in exported results.

In [ ]:
QUESTION_SETS = {
    "code": (
        ("overview", "What is the primary purpose of this codebase?"),
        (
            "mechanism",
            "How do the main components of this codebase work together?",
        ),
        (
            "limitations",
            "What limitations or validation steps are documented?",
        ),
    ),
    "literature": (
        (
            "overview",
            "What is the main research question in these documents?",
        ),
        (
            "mechanism",
            "Which principal methods are described and how are they used?",
        ),
        (
            "limitations",
            "What limitations or open questions are identified?",
        ),
    ),
}

if QUESTION_SET not in QUESTION_SETS:
    raise ValueError("Choose 'code' or 'literature'.")
if type(REPEATS) is not int or REPEATS < 1:
    raise ValueError("REPEATS must be a positive integer.")
if not TRACE_SETTINGS or any(type(x) is not bool for x in TRACE_SETTINGS):
    raise ValueError("TRACE_SETTINGS must contain booleans.")
if len(set(TRACE_SETTINGS)) != len(TRACE_SETTINGS):
    raise ValueError("TRACE_SETTINGS cannot repeat an option.")

questions = QUESTION_SETS[QUESTION_SET]
print("Question set:", QUESTION_SET)
print("Question IDs:", [name for name, _ in questions])

## 4. Optional local benchmark

The benchmark uses `kilin.RAGSession` without changing its implementation.
It creates a new conversation for each repetition and records the full
**external wall-clock** times for session startup and each answer. Trace files
remain in the benchmark workspace and are analyzed only after the session
closes. Model downloads are disabled in the workspace profile; widely used
Hugging Face offline flags are set as a second precaution. Third-party model
backends may have additional network behavior outside KíLín's control.

For trace-disabled runs, normal conversation persistence still takes place,
but diagnostic spans are unavailable. External wall time remains comparable.
The benchmark never saves full answers to its measurements or CSV output.

In [ ]:
def validate_benchmark_workspace(project: Path) -> Path:
    """Require a dedicated sandbox project with downloads disabled.

    Parameters
    ----------
    project: pathlib.Path
        Existing benchmark workspace containing a KíLín profile.

    Returns
    -------
    pathlib.Path
        Canonical benchmark project path.

    Raises
    ------
    ValueError
        If the project is outside the repository sandbox or permits downloads.
    FileNotFoundError
        If the project has no existing profile.ini.

    """

    sandbox = repository_root / "sandbox"
    if sandbox.resolve() != sandbox:
        raise ValueError("The repository sandbox must not be a symlink.")
    project = project.expanduser().resolve()
    if not project.is_relative_to(sandbox):
        raise ValueError("Use a dedicated project inside KíLín/sandbox/.")
    profile_path = project / "profile.ini"
    if not profile_path.is_file():
        raise FileNotFoundError("Create the benchmark profile.ini first.")
    profile = ConfigParser()
    profile.read(profile_path, encoding="utf-8")
    if profile.getboolean("models", "allow_download", fallback=True):
        raise ValueError(
            "Set [models] allow_download = false in the benchmark profile."
        )
    return project


def run_standard_questions(project: Path, cases: tuple):
    """Measure a fixed ordered workload in fresh KíLín conversations.

    Parameters
    ----------
    project: pathlib.Path
        Dedicated local benchmark project, already validated.
    cases: tuple
        Ordered pairs containing case identifiers and questions.

    Returns
    -------
    tuple[list[dict], dict[str, pathlib.Path]]
        Timing-only wall-clock samples and newly written diagnostic traces.

    Notes
    -----
    The saved conversation files stay in the benchmark workspace. No answers
    or question text are returned or exported. Model files must be cached.

    """

    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    import kilin

    samples = []
    traces = {}
    for enabled in TRACE_SETTINGS:
        for repetition in range(1, REPEATS + 1):
            label = f"trace{int(enabled)}_repeat{repetition:02d}"
            session = None
            try:
                started = perf_counter()
                session = kilin.RAGSession(
                    project,
                    trace=kilin.TraceConfig(enabled=enabled),
                )
                startup_ms = (perf_counter() - started) * 1000.0
                if enabled:
                    traces[label] = session.conversation_path
                for turn, (question_id, question) in enumerate(cases, 1):
                    started = perf_counter()
                    answer = session.ask(
                        question,
                        use_router=USE_ROUTER,
                    )
                    elapsed_ms = (perf_counter() - started) * 1000.0
                    samples.append({
                        "run": label,
                        "turn": turn,
                        "question_id": question_id,
                        "trace_enabled": enabled,
                        "wall_ms": elapsed_ms,
                        "startup_ms": startup_ms,
                        "answer_characters": len(answer),
                        "status": "completed",
                    })
                    print(label, question_id, f"{elapsed_ms / 1000:.2f}s")
            finally:
                if session is not None:
                    session.close()
    return samples, traces


benchmark_samples = []
generated_traces = {}
if RUN_BENCHMARK:
    benchmark_project = validate_benchmark_workspace(BENCHMARK_PROJECT)
    benchmark_samples, generated_traces = run_standard_questions(
        benchmark_project, questions
    )
else:
    print("Benchmark disabled: no models or projects were opened.")

## 5. Read saved traces and summarize turns

The parser understands KíLín's numbered `sessions`, `turns`, `stages` and
`operations`. `turn_ms` is an **inclusive** duration and must not be added to
routing or generation. Retrieval subspans can overlap their own parents;
we report individual operations rather than a misleading total. Missing
stages remain `None`. Incomplete/failed operations are not used in completed
turn summaries. A trace without diagnostic events has no stage timings.

The `run` column is a supplied label; paths and source content do not appear
in the returned operation records.

In [ ]:
trace_sources = dict(EXISTING_TRACES)
for label, path in generated_traces.items():
    if label in trace_sources:
        raise ValueError(f"Duplicate trace label: {label}")
    trace_sources[label] = path

operation_rows = []
for label, path in sorted(trace_sources.items()):
    records = read_trace_operations(path, label)
    operation_rows.extend(records)
    print(label, "timed operations:", len(records))

turn_rows = summarize_trace_turns(operation_rows)
if not trace_sources:
    print("Set EXISTING_TRACES or RUN_BENCHMARK to obtain measurements.")

for row in turn_rows:
    print(
        row["run"], "turn", row["turn"],
        "total(ms):", row["turn_ms"],
        "routing(ms):", row["routing_ms"],
        "generation(ms):", row["generation_ms"],
        "tokens/s:", row["output_tokens_per_second"],
    )

## 6. Summarize the standardized questions

The main comparison is *external wall-clock answer duration*, grouped by
question ID and trace mode. A median and observed range are shown for repeated
runs. They are descriptive, not a confidence interval; small samples cannot
establish statistically significant differences. Compare only runs with the
same project snapshot, models, settings, question order and device conditions.

For offline-only trace analysis, the previous section gives per-turn timings;
question IDs cannot be inferred reliably from arbitrary historical traces.

In [ ]:
summary_rows = question_statistics(benchmark_samples)
for row in summary_rows:
    print(
        row["question_id"], "trace:", row["trace_enabled"],
        "repeats:", row["repeats"],
        "median(ms):", round(row["median_ms"], 1),
        "range(ms):", (
            round(row["min_ms"], 1), round(row["max_ms"], 1)
        ),
    )

if summary_rows:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Install matplotlib if you want the optional chart.")
    else:
        labels = [
            f"{r['question_id']} / trace={r['trace_enabled']}"
            for r in summary_rows
        ]
        seconds = [r["median_ms"] / 1000.0 for r in summary_rows]
        _, axis = plt.subplots(figsize=(9, 4))
        axis.barh(labels, seconds)
        axis.set_xlabel("Median external answer duration (s)")
        axis.set_title("KíLín standardized-question benchmark")
        plt.tight_layout()
        plt.show()

## 7. Optional export of timing-only CSV files

Enable `EXPORT_CSV` only when you want persistent summaries. Output goes under
`KíLín/sandbox/performance_benchmark/`, excluded from the repository ZIP.
Files contain run labels, operation names, timings, token counts, and sample
status, **not** prompts, answers, source text, raw trace JSON, or paths.

In [ ]:
def write_rows(path: Path, rows: list[dict]) -> None:
    """Write timing-only dictionaries to a local CSV file.

    Parameters
    ----------
    path: pathlib.Path
        Destination path under the repository sandbox.
    rows: list[dict]
        Validated, non-sensitive performance records.

    """

    if not rows:
        return
    columns = list(dict.fromkeys(
        key for record in rows for key in record
    ))
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


if EXPORT_CSV:
    if not RESULTS_DIRECTORY.resolve().is_relative_to(
        (repository_root / "sandbox").resolve()
    ):
        raise ValueError("CSV results must stay inside sandbox/.")
    RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
    write_rows(RESULTS_DIRECTORY / "questions.csv", benchmark_samples)
    write_rows(RESULTS_DIRECTORY / "operations.csv", operation_rows)
    write_rows(RESULTS_DIRECTORY / "turns.csv", turn_rows)
    write_rows(RESULTS_DIRECTORY / "question_summary.csv", summary_rows)
    print("Timing CSV files written to:", RESULTS_DIRECTORY)

## 8. Optional, very small trace-writer check

**Not the central benchmark.** To get a rough sense of bookkeeping cost,
`TRACE_WRITE_CHECK = True` measures one batch of just 16 synthetic events
with `TraceLogger` and `NullTraceLogger`. This does not isolate disk I/O,
checkpoint persistence, warm caches, or end-to-end trace overhead. Use
`TRACE_SETTINGS = (True, False)` with real questions for the latter.

In [ ]:
def small_writer_check(event_count: int = 16) -> dict[str, float]:
    """Compare small synthetic enabled and disabled event batches.

    Parameters
    ----------
    event_count: int, optional
        Number of simple synthetic events, default 16.

    Returns
    -------
    dict[str, float]
        Elapsed milliseconds and trace file size in bytes for one batch.

    """

    import kilin

    measurements = {}
    with TemporaryDirectory() as directory:
        for enabled in (False, True):
            started = perf_counter()
            if enabled:
                logger = kilin.TraceLogger(
                    kilin.TraceConfig(
                        enabled=True,
                        output_path=Path(directory) / "conversation.json",
                    ),
                    default_directory=directory,
                )
            else:
                logger = kilin.NullTraceLogger()
            try:
                for index in range(event_count):
                    logger.emit(
                        "benchmark.synthetic",
                        component="performance",
                        data={"index": index},
                    )
            finally:
                logger.close()
            measurements[f"trace_{enabled}"] = (
                perf_counter() - started
            ) * 1000.0
            if enabled:
                measurements["trace_size_bytes"] = float(
                    (Path(directory) / "conversation.json").stat().st_size
                )
    return measurements


if TRACE_WRITE_CHECK:
    print("Illustrative, one-run trace check:", small_writer_check())

## Interpretation and limitations

- `turn_ms` includes model calls, retrieval, persistence and diagnostic
  tracing. Do **not** sum it with its constituent stages.
- `generation_ms` is the sum of completed `llm.call.output` durations, while
  output-token throughput uses recorded output counts only when available.
- Missing stage durations are **not zero**. Tool-written answers and disabled
  traces may have no LLM call; reopened traces may have multiple sessions.
- External wall time includes Python-level overhead around `ask()`. It does
  not measure GUI rendering, network access by external tools, true first-token
  latency, or operating-system cold start.
- The first run may warm caches. Treat it separately if comparing cold and
  warm behavior; new KíLín sessions do not reset operating-system caches.
- Standardized questions measure **timing**, not correctness or citation
  quality. Investigate quality separately before optimizing any operation.

See [`docs/performance.md`](../docs/performance.md) for the metric definitions
and benchmark protocol, and [`docs/trace_recording.md`](../docs/trace_recording.md)
for data sensitivity and trace persistence semantics.